# 🔄 Lensetek-Edge-Loop-3B: Multi-Variant Self-Converter

Notebook ini memungkinkan Anda mengonversi bobot dasar 16-bit (**[`lensetek/lensetek-edge-loop-3b`](https://huggingface.co/lensetek/lensetek-edge-loop-3b)**) ke varian target sesuai perangkat Anda secara gratis di Google Colab:
- **`gguf`** — Untuk Ollama, `llama.cpp`, LM Studio, Raspberry Pi, Mini PC (pilihan: `Q2_K`, `Q3_K_M`, `Q4_K_M`, `Q5_K_M`, `Q6_K`, `Q8_0`, `F16`).
- **`webgpu_onnx`** — Untuk dijalankan langsung di browser klien via **Transformers.js v3 (WebGPU `q4f16`)**.
- **`mlx_guide`** — Menghasilkan perintah 1-baris siap pakai untuk Mac Apple Silicon (M1/M2/M3/M4).

> **Tips:** Gunakan runtime **T4 GPU** atau **CPU High-RAM** di Colab (*Runtime → Change runtime type*).

In [ ]:
#@title 1. Konfigurasi Target Konversi
MODEL_ID = "lensetek/lensetek-edge-loop-3b" #@param {type:"string"}
TARGET_FORMAT = "gguf" #@param ["gguf", "webgpu_onnx", "mlx_guide"]
GGUF_QUANT = "Q4_K_M" #@param ["Q2_K", "Q3_K_M", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0", "F16"]
WEBGPU_DTYPE = "q4f16" #@param ["q4f16", "q4", "fp16"]
PUSH_TO_OWN_HF_REPO = "" #@param {type:"string"}

print(f"Model Sumber : {MODEL_ID}")
print(f"Format Target: {TARGET_FORMAT}")
if TARGET_FORMAT == "gguf":
    print(f"Kuantisasi   : {GGUF_QUANT}")
elif TARGET_FORMAT == "webgpu_onnx":
    print(f"ONNX Dtype   : {WEBGPU_DTYPE}")


In [ ]:
#@title 2. Pre-Flight Check: Validasi Ketersediaan Bobot Safetensors
import sys
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(repo_id=MODEL_ID, repo_type="model")
safetensors_files = [f for f in files if f.endswith(".safetensors")]
has_config = "config.json" in files

if not safetensors_files or not has_config:
    raise RuntimeError(
        f"[STOP] Repositori '{MODEL_ID}' belum berisi bobot 16-bit (.safetensors) & config.json.\n"
        "Silakan gunakan versi GGUF siap pakai sementara waktu di:\n"
        "https://huggingface.co/lensetek/lensetek-edge-loop-3b-GGUF"
    )

print(f"[OK] Ditemukan {len(safetensors_files)} berkas Safetensors dan config.json. Siap dikonversi!")


In [ ]:
#@title 3. Eksekusi Konversi Sesuai Format Pilihan
import os
import subprocess
from pathlib import Path

OUTPUT_ARTIFACT_PATH = None

def run_cmd(cmd):
    print(f"\n$ {cmd}")
    res = subprocess.run(cmd, shell=True)
    if res.returncode != 0:
        raise RuntimeError(f"Perintah gagal dengan exit code {res.returncode}: {cmd}")

if TARGET_FORMAT == "mlx_guide":
    print("=" * 65)
    print(" 🍎 APPLE SILICON (MLX) — JALANKAN DI TERMINAL MAC (M1/M2/M3/M4)")
    print("=" * 65)
    print("Catatan: MLX hanya dapat dijalankan di macOS Apple Silicon, bukan Linux Colab.")
    print("Salin dan jalankan perintah berikut di Terminal Mac Anda:\n")
    print("pip install -U mlx-lm")
    print(f"mlx_lm.convert --hf-path {MODEL_ID} --mlx-path ./lensetek-edge-loop-3b-mlx-4bit -q --q-bits 4")

elif TARGET_FORMAT == "gguf":
    local_hf_dir = Path("/content/hf_base_model")
    out_dir = Path("/content/gguf_output")
    out_dir.mkdir(parents=True, exist_ok=True)

    from huggingface_hub import snapshot_download
    print(f"Mengunduh {MODEL_ID} ke {local_hf_dir}...")
    snapshot_download(repo_id=MODEL_ID, local_dir=str(local_hf_dir))

    if not Path("/content/llama.cpp").exists():
        run_cmd("git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
    run_cmd("pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")

    f16_path = out_dir / "lensetek-edge-loop-3b-f16.gguf"
    run_cmd(f"python3 /content/llama.cpp/convert_hf_to_gguf.py {local_hf_dir} --outfile {f16_path} --outtype f16")

    if GGUF_QUANT.upper() == "F16":
        OUTPUT_ARTIFACT_PATH = str(f16_path)
    else:
        run_cmd("cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF && cmake --build /content/llama.cpp/build --target llama-quantize -j")
        quant_path = out_dir / f"lensetek-edge-loop-3b-{GGUF_QUANT.lower()}.gguf"
        run_cmd(f"/content/llama.cpp/build/bin/llama-quantize {f16_path} {quant_path} {GGUF_QUANT.upper()}")
        if f16_path.exists():
            f16_path.unlink()  # Hemat ruang disk Colab
        OUTPUT_ARTIFACT_PATH = str(quant_path)

    print(f"\n[SUCCESS] Berkas GGUF siap di: {OUTPUT_ARTIFACT_PATH}")

elif TARGET_FORMAT == "webgpu_onnx":
    onnx_out_dir = Path("/content/onnx_webgpu_output")
    onnx_out_dir.mkdir(parents=True, exist_ok=True)

    run_cmd("pip install -q -U transformers optimum[onnxruntime] onnxruntime-genai onnx")
    if not Path("/content/transformers.js").exists():
        run_cmd("git clone --depth 1 https://github.com/huggingface/transformers.js /content/transformers.js")

    quant_flag = "--quantize" if WEBGPU_DTYPE in ("q4f16", "q4") else ""
    run_cmd(
        f"cd /content/transformers.js && python3 -m scripts.convert --model_id {MODEL_ID} "
        f"--task text-generation-with-past --output_parent_dir {onnx_out_dir} {quant_flag}"
    )
    OUTPUT_ARTIFACT_PATH = str(onnx_out_dir)
    print(f"\n[SUCCESS] Folder ONNX WebGPU siap di: {OUTPUT_ARTIFACT_PATH}")


In [ ]:
#@title 4. (Opsional) Push Hasil ke Repo Hugging Face Pribadi atau Unduh
import os
from getpass import getpass
from huggingface_hub import HfApi

if not OUTPUT_ARTIFACT_PATH:
    print("[INFO] Tidak ada file artefak untuk diunduh/diunggah (misal mode mlx_guide).")
elif PUSH_TO_OWN_HF_REPO.strip():
    tok = ""
    try:
        from google.colab import userdata
        tok = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        pass
    if not tok:
        tok = getpass("Masukkan HF_TOKEN (Write) untuk upload hasil konversi: ").strip()

    api = HfApi(token=tok)
    api.create_repo(repo_id=PUSH_TO_OWN_HF_REPO.strip(), repo_type="model", exist_ok=True)
    if os.path.isdir(OUTPUT_ARTIFACT_PATH):
        api.upload_folder(folder_path=OUTPUT_ARTIFACT_PATH, repo_id=PUSH_TO_OWN_HF_REPO.strip(), repo_type="model")
    else:
        api.upload_file(
            path_or_fileobj=OUTPUT_ARTIFACT_PATH,
            path_in_repo=os.path.basename(OUTPUT_ARTIFACT_PATH),
            repo_id=PUSH_TO_OWN_HF_REPO.strip(),
            repo_type="model",
        )
    print(f"[OK] Berhasil diunggah ke https://huggingface.co/{PUSH_TO_OWN_HF_REPO.strip()}")
else:
    print(f"Artefak tersimpan di: {OUTPUT_ARTIFACT_PATH}")
    if os.path.isfile(OUTPUT_ARTIFACT_PATH):
        from google.colab import files
        files.download(OUTPUT_ARTIFACT_PATH)
    else:
        print("Gunakan panel Files di sebelah kiri Colab atau isi PUSH_TO_OWN_HF_REPO untuk mengambil folder hasil.")
